# Granite Speech 5.0 2B — ASR, keyword bias, and speech translation on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/granite-speech-5.0-2b-asr-translation.ipynb)

[`ibm-granite/granite-speech-5.0-2b`](https://huggingface.co/ibm-granite/granite-speech-5.0-2b) is IBM Granite's compact speech-language model (~2.3B parameters, 4.6 GB of weights, Apache 2.0). It does multilingual ASR (English, French, German, Spanish, Portuguese, Japanese) and bidirectional speech translation to and from English in those languages, plus English-to-Italian and English-to-Mandarin. `transformers>=4.52.1` loads it natively with `AutoModelForSpeechSeq2Seq` and `AutoProcessor`. The 5.0 revision adds language-ID prompting, keyword lists up to 2,000 terms, and stronger long-form translation.

This notebook is not another plain-ASR walkthrough. The [AIBackends Whisper notebook](https://github.com/donvito/notebooks/blob/main/colab/AIBackends-Whisper-audio-transcription.ipynb) already covers faster-whisper transcription plus PII redaction. The point here is Granite-specific control: **keyword biasing** (product and name lists in the prompt) and **speech translation** (X→En / En→X), plus a language-ID prompt so multilingual ASR stays in the source language instead of silently translating.

**Licence:** Apache 2.0.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has 16 GB of VRAM and no native bfloat16. The model card's example loads `torch.bfloat16`; this notebook uses **float16** on T4 (bfloat16 on L4 / A100). Weights are about 4.6 GB. Decode audio in segments of up to **~2 minutes**; split longer recordings first.

**Links:** [model card](https://huggingface.co/ibm-granite/granite-speech-5.0-2b) · [blog](https://huggingface.co/blog/ibm-granite/granite-speech-5) · [IBM docs](https://www.ibm.com/granite/docs/models/speech) · [paper](https://arxiv.org/abs/2505.08699)


In [ ]:
%pip install -q -U "transformers>=4.52.1" torchaudio soundfile

# The model card installs transformers, torchaudio, and soundfile.
# transformers>=4.52.1 is the native AutoModelForSpeechSeq2Seq / AutoProcessor floor.
# -U upgrades Colab's preinstalled transformers when it is older than that.

# espeak-ng synthesises the short keyword and French clips below (GPL, no extra Python deps).
!apt-get -qq install -y espeak-ng


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.


In [ ]:
import shutil
import subprocess
from pathlib import Path

import torch
import torchaudio
import transformers
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display
from packaging.version import Version
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

assert Version(transformers.__version__) >= Version("4.52.1"), (
    f"transformers {transformers.__version__} is loaded; granite-speech-5.0-2b needs 4.52.1+. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no native bfloat16. The card's default is bfloat16; float16 is the T4 path.
major, minor = torch.cuda.get_device_capability()
DEVICE = "cuda"
DTYPE = torch.bfloat16 if major >= 8 else torch.float16
TARGET_SR = 16_000
MAX_SECONDS = 120  # model-card segment limit

print("transformers", transformers.__version__)
print("torchaudio", torchaudio.__version__)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print("espeak-ng:", shutil.which("espeak-ng") or shutil.which("espeak") or "not found")


## 1. Sample clips

Four short clips, all resampled to 16 kHz mono:

1. the model card's own [`multilingual_sample.wav`](https://huggingface.co/ibm-granite/granite-speech-5.0-2b/blob/main/multilingual_sample.wav) (Apache 2.0, shipped in the repo)
2. a synthetic English line that names invented products (`Zyntra Pulse`, `Nexora Cloud`, `Qubitron-7`) — used for keyword biasing
3. a synthetic French line — used for language-ID ASR and French→English translation
4. a short synthetic English sentence — used for English→French translation

The product and spoken-language clips are generated with `espeak-ng` so the notebook stays license-clean and does not depend on a gated speech dataset. Robotic TTS is enough to show the prompt modes.


In [ ]:
SAMPLE_DIR = Path("granite_speech_samples")
SAMPLE_DIR.mkdir(exist_ok=True)

KEYWORD_TEXT = (
    "Please renew the Zyntra Pulse headset and the Nexora Cloud license "
    "for Qubitron-7 before Friday."
)
FRENCH_TEXT = (
    "Bonjour, je m'appelle Camille Dupont. J'habite à Lyon et je travaille "
    "chez Horizon Médical."
)
EN_TRANSLATE_TEXT = "The next design review is on Monday at three o'clock."


def load_mono16k(path):
    wav, sr = torchaudio.load(path, normalize=True)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
    return wav


def synthesize(path, text, voice):
    exe = shutil.which("espeak-ng") or shutil.which("espeak")
    if not exe:
        raise RuntimeError("espeak-ng is missing; rerun the install cell.")
    subprocess.run([exe, "-v", voice, "-s", "130", "-w", str(path), text], check=True)
    return path


official_path = Path(
    hf_hub_download(
        repo_id="ibm-granite/granite-speech-5.0-2b",
        filename="multilingual_sample.wav",
        local_dir=SAMPLE_DIR,
    )
)
keyword_path = synthesize(SAMPLE_DIR / "keyword_en.wav", KEYWORD_TEXT, "en-us")
french_path = synthesize(SAMPLE_DIR / "camille_fr.wav", FRENCH_TEXT, "fr")
en_translate_path = synthesize(SAMPLE_DIR / "review_en.wav", EN_TRANSLATE_TEXT, "en-us")

clips = {
    "official (model card)": official_path,
    "English keywords": keyword_path,
    "French": french_path,
    "English (translation source)": en_translate_path,
}

for label, path in clips.items():
    wav = load_mono16k(path)
    seconds = wav.shape[-1] / TARGET_SR
    print(f"{label}: {path.name}  {seconds:.1f}s  {tuple(wav.shape)}")
    display(Audio(wav.squeeze().numpy(), rate=TARGET_SR))


## 2. Load Granite Speech 5.0 2B

`AutoProcessor` + `AutoModelForSpeechSeq2Seq`, as on the [model card](https://huggingface.co/ibm-granite/granite-speech-5.0-2b#usage-with-transformers). The first run downloads about 4.6 GB. Prompts go through the tokenizer chat template; `<|audio|>` marks where the waveform is inserted.


In [ ]:
MODEL_ID = "ibm-granite/granite-speech-5.0-2b"

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    torch_dtype=DTYPE,
).eval()

print("dtype:", model.dtype)
print(
    f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
    f"peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)


def split_wav(wav, max_seconds=MAX_SECONDS):
    max_samples = int(max_seconds * TARGET_SR)
    length = wav.shape[-1]
    if length <= max_samples:
        return [wav]
    return [wav[:, i : i + max_samples] for i in range(0, length, max_samples)]


def run_speech(wav, instruction, max_new_tokens=200):
    """Card recipe: chat template, processor(prompt, wav), decode only new tokens."""
    user_prompt = f"<|audio|>{instruction}"
    chat = [{"role": "user", "content": user_prompt}]
    prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    pieces = []
    for chunk in split_wav(wav):
        model_inputs = processor(prompt, chunk, device=DEVICE, return_tensors="pt").to(DEVICE)
        with torch.inference_mode():
            model_outputs = model.generate(
                **model_inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=1,
            )
        num_input_tokens = model_inputs["input_ids"].shape[-1]
        new_tokens = model_outputs[0, num_input_tokens:].unsqueeze(0)
        text = tokenizer.batch_decode(
            new_tokens, add_special_tokens=False, skip_special_tokens=True
        )[0]
        pieces.append(text.strip())
    return " ".join(part for part in pieces if part)


## 3. Demo 1 — transcribe English, then French with language ID

The punctuation prompt is the card's formatted-ASR instruction. The French call prepends a language-ID sentence (`The speech is in French. Transcribe it exactly; do not translate.`) so 5.0 stays in French instead of drifting into translation. That language-ID prefix is new in 5.0.


In [ ]:
ASR_PUNCT = "transcribe the speech with proper punctuation and capitalization."
ASR_FRENCH_LID = (
    "The speech is in French. Transcribe it exactly; do not translate. "
    "can you transcribe the speech into a written format?"
)

official_wav = load_mono16k(official_path)
french_wav = load_mono16k(french_path)

print("=== official sample (ASR, punctuation) ===")
print(run_speech(official_wav, ASR_PUNCT))
print(
    f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
    f"peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)

print("\n=== French clip, language-ID prompt (expected French, not English) ===")
print("spoken:", FRENCH_TEXT)
print("transcript:", run_speech(french_wav, ASR_FRENCH_LID))


## 4. Demo 2 — keyword-biased vs unbiased transcription

Same English clip, two prompts. The unbiased call uses the card's formatted-ASR prompt. The biased call uses `transcribe the speech to text. Keywords: …` with a product/name list (up to 2,000 terms in 5.0). Watch whether `Zyntra Pulse`, `Nexora Cloud`, and `Qubitron-7` survive as written.


In [ ]:
KEYWORDS = ["Zyntra Pulse", "Nexora Cloud", "Qubitron-7"]
ASR_BIASED = "transcribe the speech to text. Keywords: " + ", ".join(KEYWORDS)

keyword_wav = load_mono16k(keyword_path)

print("spoken:", KEYWORD_TEXT)
print("keywords:", ", ".join(KEYWORDS))

print("\n=== unbiased ===")
unbiased = run_speech(keyword_wav, ASR_PUNCT)
print(unbiased)

print("\n=== keyword-biased ===")
biased = run_speech(keyword_wav, ASR_BIASED)
print(biased)


def hit_list(text):
    return [kw for kw in KEYWORDS if kw.lower() in text.lower()]


print("\nunbiased hits: ", hit_list(unbiased) or "(none exact)")
print("biased hits:   ", hit_list(biased) or "(none exact)")


## 5. Demo 3 — speech translation (French→English and English→French)

AST prompts from the card: `translate the speech to <language> with proper punctuation and capitalization.` Supported targets include English, French, German, Spanish, Japanese, Italian, and Mandarin. X→En and En→X are the trained directions (plus En→Italian / En→Mandarin).


In [ ]:
AST_TO_EN = "translate the speech to English with proper punctuation and capitalization."
AST_TO_FR = "translate the speech to French with proper punctuation and capitalization."

en_translate_wav = load_mono16k(en_translate_path)

print("=== French → English ===")
print("spoken (FR):", FRENCH_TEXT)
print("translation:", run_speech(french_wav, AST_TO_EN))

print("\n=== English → French ===")
print("spoken (EN):", EN_TRANSLATE_TEXT)
print("translation:", run_speech(en_translate_wav, AST_TO_FR))

print(
    f"\nVRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
    f"peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)


## 6. Caveats

**Two-minute segments.** The card is explicit: transcribe in chunks of up to about two minutes and split longer recordings (for example with VAD) before decoding. `run_speech` already slices at 120 seconds. The 16k YaRN context is not a licence to feed an hour of audio.

**Not the Whisper notebook.** [`AIBackends-Whisper-audio-transcription.ipynb`](https://github.com/donvito/notebooks/blob/main/colab/AIBackends-Whisper-audio-transcription.ipynb) is local faster-whisper ASR, then a PII-redaction step. It does not do keyword lists or X↔En speech translation. This notebook does not redact PII and does not use `aibackends`. Use Whisper when you want a small ASR engine inside a pipeline; use Granite Speech 5.0 2B when you want promptable ASR, language-ID, keyword biasing, or translation.

**Languages.** ASR: English, French, German, Spanish, Portuguese, Japanese. Translation: those languages to and from English, plus English→Italian and English→Mandarin. A malformed prompt falls back to plain transcription.

**T4 / dtype.** Float16 on T4; bfloat16 on GPUs with native bf16. Peak VRAM after a short clip should stay well under 16 GB (~4.6 GB weights plus encoder/decoder activations).

**Synthetic clips.** `espeak-ng` is robotic. It is here so the keyword and French demos stay license-clean and offline-reproducible. Swap in your own 16 kHz mono recordings for a fairer quality check.
